In [ ]:
# Install packages
%pip install langchain-deepseek
%pip install langsmith

In [ ]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from google.colab import userdataΩΩΩ

In [ ]:
# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT')
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
# Test the output
userdata.get('LANGSMITH_PROJECT')


We will start exploring the use of AI Agents using the core components of messages.

Messages are the fundamental unit of context for models in LangChain.

Here the unit is an object that contains information about:

- The role (whether system or user)
- Content (actual contents of the message)
- Metadata (response information)

They represent the input, metadata, and output of the conversation when interacting with an llm.

In [ ]:
# Initialise the LLM Model
model = init_chat_model("deepseek:deepseek-v4-flash")

Message content is a data payload that gets sent to and from the model with a number of attributes.

The input into the model, is a **HumanMessage** object, which represents the user's input.


The output of the model, is a **AIMessage** objecttype, which represents the model's output.

A **SystemMessage** represents an initial set of instructions that primes the model's behaviour. Use it to set the tone, define the model's role, and establish guidelines for responses.

The system message is essentially the object payload to deliver the prompt, which can be a superior way to do so compared to a simple string. This gives you more control over the primpt structure, which can be useful for some models.

In [ ]:
system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": """You are a medieval poet and you only answer in the form of limericks.""",
            },
            {
                "type": "text",
                "text": "Specifically a Viking poet"
            }
        ]
    )

messages = [
    system_prompt,
    HumanMessage("What is the capital of France?")
]


Message objects implement a content_blocks property. This parses the content attribute of into a standard representation.

In [ ]:
## Interact with the LLM Model
response = model.invoke(messages)
print(response)

In [ ]:
print(f"Response contents: {response.content_blocks}")
print(f"Data type of the response conents: {type(response.content_blocks)}")
print("")
print(f"Data type of the first layer of the response contents: {type(response.content_blocks[0])}")
print("")
print(response.content_blocks[0]["text"])

In [ ]:
# More metadata fun
print(response.usage_metadata)

### **Parameters, parameters, and more darn parameters**

We can set the behaviour of our model at initialisation.

Some parameters  of the **init_chat_model** function we will look at are:

**temperature:** Between 0-2.This controls the randomness of the model output. The lower the number more deterministic.

**max_tokens:** Limits the number of tokens in the response. Mo' Tokens Mo' Money

**timeout:** The maximum time (in seconds) to wait for a response.

**max_retries**: Maximum time (in seconds) to wait for a response from the model before canceling request.

**Exercise:** Experiment with some of the parameters to control the output of the model.

Get really creative!

In [ ]:
# Initialise the LLM Model
model = init_chat_model(model = "deepseek:deepseek-v4-flash",
                        temperature = 2,
                        max_tokens = 200)
system_msg = SystemMessage("""You are a caveman and only respond in short terse expressions.""")

messages = [
    system_msg,
    HumanMessage("What is the cause of economic inequality?")
]

# Interact with the LLM Model
response = model.invoke(messages)
print(response.content_blocks[0]["text"])

## **init_chat_model and create_agent**
What we just used was a simple model intialiser,  which returns a preconfigured chat model instance with no tools, loops.

create_agent() builds an agent runtime that supports:


*   Static or dynamic prompts
*   Static or dynamic tools
*   Middleware
*   Checkpointing, memory





**Create an agent with dynamic prompting**

In [ ]:
# Import packages
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

# Set the dynamic prompt using @dynamic_prompt decorator
@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")

    base_prompt = "You are a helpful assistant." # Your foundational prompt regardless of context.

    if user_role == "experienced":
        return f"{base_prompt} Explain it to me like an adult"

    elif user_role == "beginner":
        return f"{base_prompt} Explain it to me like I'm 5 years old."

    return base_prompt

In [ ]:
agent = create_agent(
    model="deepseek:deepseek-v4-flash",
    middleware=[user_role_prompt]
)

# The system prompt will be set dynamically based on context
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What meaning of 'Don't start none, won't be none' and where does it originate?"}]},
    context={"user_role":"beginner"}
)

In [ ]:
# View results
print(response)

In [ ]:
ai_messages = [
    m for m in response["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
output_list

In [ ]:
# If you want a neater output
print(output_list[0][0]["text"])